# 04 — Model Comparison & Statistical Testing
Evaluates every trained model on the held-out test set (Section 8), runs
stratified cross-validation (Section 9), and applies paired statistical
tests (Wilcoxon, McNemar, DeLong) plus a bootstrap confidence interval to
determine whether the observed differences are statistically meaningful.


In [1]:
# ---------------------------------------------------------------------------
# Make the src/ package importable from inside notebooks/, and fix all
# random seeds so this notebook is reproducible (Section 14 requirement).
# ---------------------------------------------------------------------------
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import preprocessing, features, models, evaluation, explainability

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option("display.max_columns", 50)
plt.rcParams["figure.figsize"] = (8, 4)


In [2]:
import joblib

X_train = pd.read_csv("../data/processed/X_train.csv")
X_test = pd.read_csv("../data/processed/X_test.csv")
y_train = pd.read_csv("../data/processed/y_train.csv").squeeze("columns")
y_test = pd.read_csv("../data/processed/y_test.csv").squeeze("columns")

trained_supervised = {
    strategy: {
        name: joblib.load(f"../models/{name}_{strategy}.joblib")
        for name in ["logistic_regression", "random_forest", "xgboost"]
    }
    for strategy in models.IMBALANCE_STRATEGIES
}
iso_forest = joblib.load("../models/isolation_forest.joblib")
autoencoder = joblib.load("../models/autoencoder.joblib")


In [3]:
# ---------------------------------------------------------------------------
# Section 8: evaluate every model x strategy combination on the test set
# ---------------------------------------------------------------------------
all_results = []
for strategy, model_dict in trained_supervised.items():
    for model_name, model_obj in model_dict.items():
        preds = model_obj.predict(X_test)
        scores = model_obj.predict_proba(X_test)[:, 1]
        all_results.append(evaluation.compute_metrics(
            y_test, preds, scores, name=f"{model_name} ({strategy})"
        ))

iso_pred, iso_score = models.isolation_forest_scores(iso_forest, X_test)
all_results.append(evaluation.compute_metrics(y_test, iso_pred, iso_score, name="isolation_forest"))

ae_pred, ae_score = models.autoencoder_scores(autoencoder, X_test, float(y_train.mean()))
all_results.append(evaluation.compute_metrics(y_test, ae_pred, ae_score, name="autoencoder"))

results_table = evaluation.metrics_table(all_results)
results_table.sort_values("f1", ascending=False)


,model,precision,recall,f1,roc_auc,pr_auc
5,xgboost (class_weight),0.948718,0.778947,0.855491,0.977166,0.827412
1,random_forest (none),0.972603,0.747368,0.845238,0.944013,0.800645
4,random_forest (class_weight),0.958904,0.736842,0.833333,0.953448,0.816301
7,random_forest (smote),0.910256,0.747368,0.820809,0.974382,0.822493
2,xgboost (none),0.957143,0.705263,0.812121,0.971759,0.807724
8,xgboost (smote),0.681818,0.789474,0.731707,0.971402,0.808845
0,logistic_regression (none),0.848485,0.589474,0.695652,0.958099,0.691091
10,autoencoder,0.389474,0.389474,0.389474,0.953904,0.341849
9,isolation_forest,0.222222,0.252632,0.236453,0.935134,0.117254
3,logistic_regression (class_weight),0.055333,0.873684,0.104075,0.968489,0.710525


## Experiment A vs B vs C — does imbalance handling matter?
Compare the SAME model (XGBoost) across the three strategies directly.


In [4]:
imbalance_comparison = evaluation.metrics_table([
    r for r in all_results if r["model"].startswith("xgboost")
])
imbalance_comparison


,model,precision,recall,f1,roc_auc,pr_auc
0,xgboost (none),0.957143,0.705263,0.812121,0.971759,0.807724
1,xgboost (class_weight),0.948718,0.778947,0.855491,0.977166,0.827412
2,xgboost (smote),0.681818,0.789474,0.731707,0.971402,0.808845


## Stratified cross-validation (Section 9)
Single train/test splits can be misleading. We cross-validate the two
strongest supervised models (Random Forest, XGBoost, both under the SMOTE
strategy) to get a distribution of scores rather than one number each.


In [5]:
def make_imbalance_fn(strategy):
    return lambda Xtr, ytr: models.apply_imbalance_strategy(Xtr, ytr, strategy)

rf_cv_scores = evaluation.stratified_cv_f1(
    lambda: models.build_random_forest("smote"), make_imbalance_fn("smote"),
    X_train, y_train, n_splits=5
)
xgb_cv_scores = evaluation.stratified_cv_f1(
    lambda: models.build_xgboost("smote"), make_imbalance_fn("smote"),
    X_train, y_train, n_splits=5
)
print("Random Forest CV F1:", np.round(rf_cv_scores, 3),
      f"(mean {np.mean(rf_cv_scores):.3f} +/- {np.std(rf_cv_scores):.3f})")
print("XGBoost CV F1:      ", np.round(xgb_cv_scores, 3),
      f"(mean {np.mean(xgb_cv_scores):.3f} +/- {np.std(xgb_cv_scores):.3f})")


Random Forest CV F1: [0.853 0.869 0.803 0.855 0.886] (mean 0.853 +/- 0.028)
XGBoost CV F1:       [0.819 0.813 0.739 0.832 0.8  ] (mean 0.801 +/- 0.033)


## Paired statistical tests
Three complementary tests, as recommended in the methodology:
- **Wilcoxon signed-rank test** on the CV fold scores above.
- **McNemar's test** on the paired test-set predictions (same test set,
  two models).
- **DeLong's test** comparing the two models' ROC-AUCs directly.
- A **bootstrap confidence interval** around XGBoost's ROC-AUC.


In [6]:
try:
    stat, p_wilcoxon = evaluation.wilcoxon_test(rf_cv_scores, xgb_cv_scores)
    print(f"Wilcoxon: statistic={stat:.3f}, p={p_wilcoxon:.4f}")
except Exception as e:
    print("Wilcoxon test could not be computed on this data:", e)


Wilcoxon: statistic=0.000, p=0.0625


In [7]:
rf_smote = trained_supervised["smote"]["random_forest"]
xgb_smote = trained_supervised["smote"]["xgboost"]

rf_pred = rf_smote.predict(X_test)
xgb_pred = xgb_smote.predict(X_test)

mcnemar_result, table = evaluation.mcnemar_test(y_test, xgb_pred, rf_pred)
print("2x2 table [[both correct, xgb-only correct],[rf-only correct, both wrong]]:")
print(np.array(table))
print(f"McNemar statistic={mcnemar_result.statistic:.3f}, p={mcnemar_result.pvalue:.4f}")


2x2 table [[both correct, xgb-only correct],[rf-only correct, both wrong]]:
[[56686     5]
 [   29    26]]
McNemar statistic=15.559, p=0.0001


In [8]:
rf_score = rf_smote.predict_proba(X_test)[:, 1]
xgb_score = xgb_smote.predict_proba(X_test)[:, 1]

auc_xgb, auc_rf, z_stat, p_delong = evaluation.delong_roc_test(y_test.values, xgb_score, rf_score)
print(f"XGBoost ROC-AUC={auc_xgb:.4f}, Random Forest ROC-AUC={auc_rf:.4f}")
print(f"DeLong z={z_stat:.3f}, p={p_delong:.4f}")


XGBoost ROC-AUC=0.9714, Random Forest ROC-AUC=0.9744
DeLong z=-0.233, p=0.8159


In [9]:
point, lo, hi = evaluation.bootstrap_ci(y_test.values, xgb_score, n_boot=1000)
print(f"XGBoost ROC-AUC = {point:.4f}  (95% bootstrap CI: [{lo:.4f}, {hi:.4f}])")


XGBoost ROC-AUC = 0.9714  (95% bootstrap CI: [0.9496, 0.9894])


**Reporting guidance:** state the winning model, its CV mean +/- std, and
which of the paired tests (if any) reached p < 0.05 -- and be explicit
about sample-size limits if the fraud class in your test set is small,
since that limits how much statistical power these tests have.
